# 📑 Hoist Monthly Report
 
## 🔗 Links
- **[BRD - Hoist Monthly Report](https://cepalhellas.atlassian.net/wiki/spaces/DBTC/pages/308379650/Updated+BRD+Hoist+Reporting+-+Line+by+Line+LbL)**  
  Comprehensive documentation outlining the business requirements for this report.
 
- **[Story Ticket](https://cepalhellas.atlassian.net/browse/DBTC-168)**  
  Details about the story ticket, including tasks and progress.
 
- **[UAT Findings Ticket](https://cepalhellas.atlassian.net/browse/DBTC-516)**  
  A list of findings from UAT.
 
---

## Initialization

In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets
def get_aws_credentials(secrets_scope, key_name):
    try:
        # Databricks secrets configuration
    
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        # if aws_secret_key is not None:
        #     print("AWS secret key retrieved successfully.")
        # else:
        #     print("AWS secret key is None. Please check Databricks secrets configuration.")

        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration

        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    print("mode is shared")
    # Connect to AWS using boto3
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')


In [0]:
from typing import Any
import shutil
import os

from delta import DeltaTable  # type: ignore
from pyspark.sql.functions import (
    coalesce,
    col,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    trim,
    when,
    months_between,
    dateadd,
    datediff,
    regexp_replace
)

from typing import Optional

import delta  # type: ignore
from pyspark.sql import SparkSession

from io import BytesIO
import pandas as pd
import json

from datetime import datetime
from pytz import timezone

In [0]:
athens_tz = timezone('Europe/Athens')

try:    
    report_args = dbutils.widgets.get("report_args")
    report_args = report_args.replace("'", '"')
except Exception as e:
    raise ValueError("Missing required input: report_args")

try:    
    report_name = dbutils.widgets.get("report_name")
except Exception as e:
    raise ValueError("Missing required input: report_name")
print(f"Report {report_name} is started")

try:
    run_date = dbutils.widgets.get("run_date")
    if run_date is None or run_date == "": raise ValueError("ERROR: run_date has no value. Remove the run_date completely if you want to run the job using the current date or add a value in the following format 'YYYY-MM-DD'")
    run_date_str = run_date.replace('-', '')
    run_date = datetime.strptime(run_date, '%Y-%m-%d').date()
except Exception as e:
    run_date = datetime.now(athens_tz).strftime('%Y-%m-%d')
    run_date_str = datetime.now(athens_tz).strftime('%Y%m%d')

now_datetime_str = datetime.now(athens_tz).strftime('%Y%m%dT%H%M%S')

run_date = run_date.strftime('%Y-%m-%d')

# Read JSON
json_data = json.loads(report_args)
 
# Access the values and store them into variables
if "portfolio_name" not in json_data: raise KeyError("'portfolio_name' key is missing in the report_args JSON data.")
if "output_file_prefix" not in json_data: raise KeyError("'output_file_prefix' key is missing in the report_args JSON data.")
if "report_output" not in json_data: raise KeyError("'report_output' key is missing in the report_args JSON data.")

portfolioName = "".join(json_data["portfolio_name"])
output_file_prefix = "".join(json_data["output_file_prefix"])
report_output = "".join(json_data["report_output"])

print(f"Report {report_name} is running with the following arguments: report_args = {report_args}, report_name = {report_name}, portfolio_name = {portfolioName}, output_file_prefix = {output_file_prefix}, run_date = {run_date}")

print(f"now is: {now_datetime_str}")

In [0]:
migrationDate = spark.sql(f"""
    SELECT MIN(CAST(AssignmentDate AS DATE)) AS migrationDate
    FROM `{env_prefix}-cdh`.primary.portfolio
    WHERE PortfolioName = {portfolioName!r}
""").collect()[0][0]

eomDate = spark.sql(f"""SELECT DATE_ADD(DATE_TRUNC('month', '{run_date}'), -1)""").collect()[0][0]
eomDate = eomDate.strftime('%Y-%m-%d')

RefMonth = spark.sql(f"SELECT CAST(DATEADD(DAY, 1 - DAY('{eomDate}'), CAST('{eomDate}' AS DATE)) AS DATE)").collect()[0][0]

FirstWorkingDate = spark.sql(f"""
    SELECT MIN(CalendarDate)
    FROM `{env_prefix}-cdh`.reference.ref_calendar
    WHERE FirstDateOfMonth = '{RefMonth}' AND IsBusinessDay = 1
""").collect()[0][0]

LastWorkingDate = spark.sql(f"""
    SELECT MAX(CalendarDate)
    FROM `{env_prefix}-cdh`.reference.ref_calendar
    WHERE FirstDateOfMonth = '{RefMonth}' AND IsBusinessDay = 1
""").collect()[0][0]

## Loading Tables

In [0]:
dexposure = spark.sql(f"""
                      SELECT *,
                      {run_date!r} AS RunDate
                      FROM `{env_prefix}-cdh`.derivative.get_all_exposures_by_business_date(DATE({LastWorkingDate!r}))
                      WHERE sourcesystem = 'profits' AND PortfolioName = {portfolioName!r}
                      """)

dexposure.createOrReplaceTempView("dexposure")

dparty_l3869 = spark.sql(f"""
                    select CDI, MAX(CASE WHEN BusinessDate < '{LastWorkingDate}' AND L3869Flag IS TRUE THEN 1 ELSE 0 END) AS L3869PrevFlag
                    from `{env_prefix}-cdh`.derivative.dparty
                    WHERE sourcesystem='profits' AND PortfolioName = {portfolioName!r}
                    GROUP BY CDI
                     """)

dparty_l3869.createOrReplaceTempView("dparty_l3869")

dparty = spark.sql(f"""
                    select DISTINCT A.CDI, L3869Flag, Segment,
                    CASE WHEN L3869Flag IS TRUE THEN 'Under L3869'
                    WHEN L3869Flag IS FALSE AND L3869PrevFlag = 0 THEN 'Not Under L3869'
                    WHEN L3869Flag IS FALSE AND L3869PrevFlag = 1 THEN 'Exit L3869'
                    ELSE 'N/A' END AS L3869Status
                    from `{env_prefix}-cdh`.derivative.dparty A
                    LEFT JOIN dparty_l3869 B
                    ON A.CDI = B.CDI
                    WHERE BusinessDate = {LastWorkingDate!r}
                    and sourcesystem='profits' AND PortfolioName = {portfolioName!r}
                     """)

dparty.createOrReplaceTempView("dparty")

dunderlaw_active = spark.sql(f"""
                    SELECT *
FROM (
    SELECT 
            CDI,
            InLawFlag,
            UnderLawType,
            BusinessDate,
            ApplicationStatusDate,
            ROW_NUMBER() OVER (
                    PARTITION BY CDI
                    ORDER BY
                        CASE 
                          WHEN UnderLawType = 'N4738/2020' and LawSystem = 'Βεβαίωση ευάλωτου' THEN 1
                          WHEN UnderLawType = 'N4738/2020' and LawSystem = 'Πτώχευση' THEN 2
                          WHEN UnderLawType = 'N4738/2020' and LawSystem = 'Εξυγίανση' THEN 3
                          WHEN UnderLawType = 'N4738/2020' and LawSystem = 'Εξωδικαστικός μηχανισμός' THEN 4
                          WHEN UnderLawType = 'N3869'  THEN 5
                          WHEN UnderLawType = 'N4605/2019' THEN 6
                          ELSE 7
                        END
            ) AS rn
    FROM `{env_prefix}-cdh`.derivative.dunderlaw
    WHERE BusinessDate = '{LastWorkingDate}'
      AND PortfolioName = {portfolioName!r}
      AND ApplicationStatus NOT IN ('Rejected', 'Resigned')
) t
WHERE rn = 1; 
                     """)

dunderlaw_active.createOrReplaceTempView("dunderlaw_active")

settlementexposure = spark.sql(f"""
                    select DISTINCT *
                    from `{env_prefix}-cdh`.primary.settlementexposure
                    WHERE BusinessDate = {LastWorkingDate!r}
                    AND PortfolioName = {portfolioName!r}
                    and isActiveFlag=1
                    AND IsMigratedFlag = False
                    and isPrimarySettlementFlag = True
                     """)

settlementexposure.createOrReplaceTempView("settlementexposure")

drestructuring = spark.sql(f"""
                    select DISTINCT *
                    from `{env_prefix}-cdh`.derivativeactive.drestructuring
                    WHERE BusinessDate = {LastWorkingDate!r}
                    AND PortfolioName = {portfolioName!r}
                    AND RestructuringType IS NOT NULL
                     """)

drestructuring.createOrReplaceTempView("drestructuring")

dportfoliotoolthirdparties = spark.sql(f"""
                    select DISTINCT *
                    from `{env_prefix}-cdh`.derivativeactive.dportfoliotoolthirdparties
                    WHERE BusinessDate = {LastWorkingDate!r}
                    AND PortfolioName = {portfolioName!r}
                    AND frequencytype='Monthly'
                     """)
 
dportfoliotoolthirdparties.createOrReplaceTempView("dportfoliotoolthirdparties")

## External Files

In [0]:
party = spark.sql(f"""
                      select DISTINCT PortfolioName, TaxRegistrationNumber, subsegment
                      from `{env_prefix}-cdh`.primary.party
                      where BusinessDate <= {LastWorkingDate!r}
                      AND PortfolioName = {portfolioName!r}
                      and sourcesystem='qc'
                      """)

party.createOrReplaceTempView("party")

LetterFlag = spark.sql(f"""
                      select DISTINCT AFM
                      from `{env_prefix}-ReportInputFiles`.investor.mercury_letterflag
                      where to_date(IngestionTimestamp, 'd/M/yyyy') = (SELECT to_date(MAX(IngestionTimestamp), 'd/M/yyyy') FROM `{env_prefix}-ReportInputFiles`.investor.mercury_letterflag)
                      """)

LetterFlag.createOrReplaceTempView("LetterFlag")

## Collateral fields

In [0]:
dcollateral = spark.sql(f"""
                        select * 
                        from `{env_prefix}-cdh`.derivative.dcollateral
                        where BusinessDate = {LastWorkingDate!r} and PortfolioName = {portfolioName!r}
                      """)

dcollateral.createOrReplaceTempView("dcollateral")

realestateinfo = spark.sql(f"""
                           select * 
                           from `{env_prefix}-cdh`.primary.realestateinfo
                           where BusinessDate = {LastWorkingDate!r} and SourceSystem = 'thesis'
                           """)

realestateinfo.createOrReplaceTempView("realestateinfo")

drealestate = spark.sql(f"""
                        select *
                        from `{env_prefix}-cdh`.derivative.drealestate
                        where BusinessDate = {LastWorkingDate!r}
                      """)

drealestate.createOrReplaceTempView("drealestate")

In [0]:
CollateralMapping = spark.sql(f"""
    SELECT DISTINCT
        dcol.BusinessDate,
        dcol.PortfolioName,
        dre.RealEstateCepalKey,
        re.OwnershipKey,
        dex.ExposureKey,
        dcol.AgreementKey,
        dcol.DCollateralKey,
        dcol.CollateralAmount,
        dcol.CollateralType,
        dcol.CollateralTypeDescription,
        dcol.InitialLegalOrder,
        dcol.RealEstateCollateralStatus,
        case when CollateralType ='RE' THEN REMortgageRegistrationDate ELSE NULL END REMortgageRegistrationDate,
        COALESCE(case when CollateralType = 'RE' THEN dcol.RealEstateCollateralStatus ELSE dcol.CollateralStatus END, 'Inactive') CollateralStatus,
        dre.RealEstateActive,
        case when dcol.CollateralType ='RE' then (
        case when dcol.subcategory = 'Προσημείωση' THEN 'Prenotation' 
        else 'Mortgage' end) 
        else Null END AS LienType,
        dex.CDI,
        dcol.Currency
    FROM dcollateral dcol
        LEFT JOIN realestateinfo re
        ON dcol.RealEstateKey = re.RealEstateKey
        AND re.OwnerPartyKey = dcol.OwnerPartyKey
        LEFT JOIN dexposure dex
        ON dex.AgreementKey = dcol.AgreementKey
        LEFT JOIN drealestate dre
        ON dcol.RealEstateKey = dre.RealEstateKey
    WHERE dcol.CollateralType is not null
""")

CollateralMapping.createOrReplaceTempView('CollateralMapping')

In [0]:
RankedAgreements = spark.sql(f"""
                        SELECT ExposureKey, AgreementKey, DCollateralKey,
                        InitialLegalOrder,
                        SUM(CollateralAmount) OVER (PARTITION BY ExposureKey, AgreementKey) AS CollateralTotalAmount,
                        SUM(CollateralAmount) OVER (PARTITION BY ExposureKey, AgreementKey, InitialLegalOrder) AS CollateralLienTotalAmount,
                        ROW_NUMBER() OVER (PARTITION BY ExposureKey, AgreementKey ORDER BY InitialLegalOrder ASC, DCollateralKEY ASC) AS Rank,
                        MIN(CollateralAmount) OVER (PARTITION BY ExposureKey, AgreementKey) AS MinCollateralTotalAmount
                        FROM CollateralMapping
                        WHERE CollateralStatus = 'Active'
""")

RankedAgreements.createOrReplaceTempView("RankedAgreements")

CollateralMappingTempLien = spark.sql(f"""
                      SELECT ExposureKey, AgreementKey, DCollateralKey,
                            InitialLegalOrder AS MinCollateralLien,
                            CollateralTotalAmount,
                            CASE WHEN InitialLegalOrder IS NULL THEN MinCollateralTotalAmount ELSE CollateralLienTotalAmount END AS MinCollateralLienTotalAmount
                        FROM RankedAgreements
                        WHERE Rank = 1
""")

CollateralMappingTempLien.createOrReplaceTempView("CollateralMappingTempLien")

In [0]:
# %sql
# SELECT *
# from CollateralMappingTempLien
# where ExposureKey = 'MRC989002044961484'

## Initial Assignment Amount

In [0]:
# We use the InitialAssignmentAmount column from dexposure
dexposure_filtered = spark.sql(f"""
                    SELECT DISTINCT
                        A.ExposureKey,
                        A.OriginationAmount,
                        A.OutstandingTotalBalance,
                        A.InitialAssignmentAmount,
                        A.BusinessDate
                    FROM `{env_prefix}-cdh`.derivative.dexposure A
                    INNER JOIN dexposure B
                    ON A.ExposureKey = B.ExposureKey
""")

dexposure_filtered.createOrReplaceTempView("dexposure_filtered")

dexposure_at_migration = spark.sql(f"""
                    SELECT ExposureKey, MIN(BusinessDate) AS MinDate
                    FROM dexposure_filtered
                    WHERE BusinessDate >= '{migrationDate}'
                    GROUP BY ExposureKey
""")

dexposure_at_migration.createOrReplaceTempView("dexposure_at_migration")

dinitTLC = spark.sql(f"""
                    SELECT DISTINCT
                        A.ExposureKey,
                        OutstandingTotalBalance AS TLCAtMigrationDate
                    FROM dexposure_filtered A
                    INNER JOIN dexposure_at_migration B
                    ON A.ExposureKey = B.ExposureKey AND A.BusinessDate = B.MinDate
""")

dinitTLC.createOrReplaceTempView("dinitTLC")

In [0]:
# %sql
# SELECT *
# FROM dinitTLC
# WHERE ExposureKey IN ('MRC36711951190008', 'MRC989001864343281')

## Balance Range

In [0]:
CDITaxNumber = spark.sql(f"""
                SELECT DISTINCT PortfolioName, CDI,
                        CASE WHEN PartyTaxRegistrationNumber IS NOT NULL THEN PartyTaxRegistrationNumber ELSE CDI END AS CDITaxNumber,
                        OutstandingTotalBalance, InitialAssignmentAmount
                FROM dexposure
                WHERE OutstandingTotalBalance IS NOT NULL OR InitialAssignmentAmount IS NOT NULL
""")

CDITaxNumber.createOrReplaceTempView("CDITaxNumber")

TaxBalance = spark.sql(f"""
                SELECT PortfolioName, CDITaxNumber, SUM(OutstandingTotalBalance) AS OutstandingTotalBalance, SUM(InitialAssignmentAmount) AS InitialAssignmentAmount
                FROM CDITaxNumber
                GROUP BY PortfolioName, CDITaxNumber
""")

TaxBalance.createOrReplaceTempView("TaxBalance")

BalanceRange = spark.sql(f"""
                            SELECT CDITaxNumber,PortfolioName,
                                CASE
                                WHEN PortfolioName = 'Mercury' AND OutstandingTotalBalance < 5000 THEN CONCAT(PortfolioName, ' < 5k')
                                WHEN PortfolioName = 'Mercury' AND (OutstandingTotalBalance >= 5000 AND OutstandingTotalBalance <= 10000) THEN CONCAT(PortfolioName, ' 5k - 10k')
                                WHEN PortfolioName = 'Mercury' AND (OutstandingTotalBalance > 10000 AND OutstandingTotalBalance <= 20000) THEN CONCAT(PortfolioName, ' 10k - 20k')
                                WHEN PortfolioName = 'Mercury' AND (OutstandingTotalBalance > 20000 AND OutstandingTotalBalance <= 50000) THEN CONCAT(PortfolioName, ' 20k - 50k')
                                WHEN PortfolioName = 'Mercury' AND (OutstandingTotalBalance > 50000 AND OutstandingTotalBalance <= 75000) THEN CONCAT(PortfolioName, ' 50k - 75k')
                                WHEN PortfolioName = 'Mercury' AND (OutstandingTotalBalance > 75000 AND OutstandingTotalBalance <= 125000) THEN CONCAT(PortfolioName, ' 75k - 125k')
                                WHEN PortfolioName = 'Mercury' AND (OutstandingTotalBalance > 125000 AND OutstandingTotalBalance <= 200000) THEN CONCAT(PortfolioName, ' 125k - 200k')
                                WHEN PortfolioName = 'Mercury' AND OutstandingTotalBalance > 200000 THEN CONCAT(PortfolioName, ' 200k+')
                                WHEN PortfolioName != 'Mercury' AND InitialAssignmentAmount < 5000 THEN CONCAT(PortfolioName, ' < 5k')
                                WHEN PortfolioName != 'Mercury' AND (InitialAssignmentAmount >= 5000 AND InitialAssignmentAmount <= 10000) THEN CONCAT(PortfolioName, ' 5k - 10k')
                                WHEN PortfolioName != 'Mercury' AND (InitialAssignmentAmount > 10000 AND InitialAssignmentAmount <= 20000) THEN CONCAT(PortfolioName, ' 10k - 20k')
                                WHEN PortfolioName != 'Mercury' AND (InitialAssignmentAmount > 20000 AND InitialAssignmentAmount <= 50000) THEN CONCAT(PortfolioName, ' 20k - 50k')
                                WHEN PortfolioName != 'Mercury' AND (InitialAssignmentAmount > 50000 AND InitialAssignmentAmount <= 75000) THEN CONCAT(PortfolioName, ' 50k - 75k')
                                WHEN PortfolioName != 'Mercury' AND (InitialAssignmentAmount > 75000 AND InitialAssignmentAmount <= 125000) THEN CONCAT(PortfolioName, ' 75k - 125k')
                                WHEN PortfolioName != 'Mercury' AND (InitialAssignmentAmount > 125000 AND InitialAssignmentAmount <= 200000) THEN CONCAT(PortfolioName, ' 125k - 200k')
                                ELSE CONCAT(PortfolioName, ' 200k+') END AS BalanceRange
                            FROM TaxBalance
""")

BalanceRange.createOrReplaceTempView("BalanceRange")

TaxBalanceRange = spark.sql(f"""
    SELECT DISTINCT A.CDI, B.BalanceRange
    FROM CDITaxNumber A
    INNER JOIN BalanceRange B
    ON A.CDITaxNumber = B.CDITaxNumber
       AND A.PortfolioName = B.PortfolioName
    WHERE A.OutstandingTotalBalance IS NOT NULL OR A.InitialAssignmentAmount IS NOT NULL
""")


TaxBalanceRange.createOrReplaceTempView("TaxBalanceRange")

## Settlement fields

### Implementation Date

In [0]:
sttlexp_impltdate = spark.sql(f"""
                    SELECT DISTINCT A.ExposureKey, CAST(B.ImplementationDate AS DATE) AS ImplementationDate,
                    B.SettlementStatus, CAST(B.SubmissionDate AS DATE) AS SubmissionDate, CAST(A.ClosedDate AS DATE) AS ClosedDate
                    FROM dexposure A
                    LEFT JOIN settlementexposure B
                    ON regexp_replace(A.ExposureKey, '[^0-9]', '') = regexp_replace(B.ExposureKey, '[^0-9]', '')
                    AND regexp_replace(A.CDI, '[^0-9]', '') = regexp_replace(B.CDI, '[^0-9]', '')
                    AND A.PortfolioName = B.PortfolioName
                    and isActiveFlag=1
                    AND IsMigratedFlag = False
                    and isPrimarySettlementFlag
""")

sttlexp_impltdate.createOrReplaceTempView("sttlexp_impltdate")

paymentschedule_impltdate = spark.sql(f"""
                    SELECT A.ExposureKey, MIN(CAST(B.InstallmentDate AS DATE)) AS InstallmentDate
                    FROM dexposure A
                    LEFT JOIN `{env_prefix}-cdh`.derivative.dpaymentschedule B
                    ON regexp_replace(A.ExposureKey, '[^0-9]', '') = regexp_replace(B.ExposureKey, '[^0-9]', '')
                    AND A.PortfolioName = B.PortfolioName
                    WHERE B.BusinessDate = {LastWorkingDate!r}
                    GROUP BY A.ExposureKey
""")

paymentschedule_impltdate.createOrReplaceTempView("paymentschedule_impltdate")

drstr_date = spark.sql(f"""
                    SELECT DISTINCT NewExposureKey AS ExposureKey, RestructuringDate
                    FROM drestructuring
""")

drstr_date.createOrReplaceTempView("drstr_date")

In [0]:
impl_date = spark.sql(f"""
                    SELECT DISTINCT ExposureKey, CASE WHEN RestructuringDate > ImplementedDate THEN RestructuringDate ELSE ImplementedDate END as ImplementedDate
                    FROM
                    (
                    SELECT DISTINCT A.*, InstallmentDate, RestructuringDate,
                            CASE WHEN ImplementationDate IS NULL AND SettlementStatus = 'Out of Collection'
                            THEN CASE WHEN InstallmentDate IS NOT NULL THEN InstallmentDate
                                    WHEN InstallmentDate IS NULL AND SubmissionDate IS NOT NULL THEN SubmissionDate
                                    ELSE ClosedDate END ELSE ImplementationDate END AS ImplementedDate
                    FROM sttlexp_impltdate A
                    LEFT JOIN paymentschedule_impltdate B
                    ON A.ExposureKey = B.ExposureKey
                    LEFT JOIN drstr_date C
                    ON A.ExposureKey = C.ExposureKey
                    )
""")

impl_date.createOrReplaceTempView("impl_date")

# number of implementations per account where we need to use Implemented Date as will be calculated in CUD

# if Mercury then LTSA is the cut-off date else it is the migration date
if portfolioName == 'Mercury':
    cutoffdate = '2019-07-01'
elif portfolioName in ('Orbit', 'Light'):
    cutoffdate = '2022-03-28'
elif portfolioName == 'Cell':
    cutoffdate = '2023-10-23'
elif "pearl" in portfolioName.lower():
    cutoffdate = '2024-02-19'
else:
    cutoffdate = migrationDate  

impl_count = spark.sql(f"""
                    SELECT A.ExposureKey, COUNT(DISTINCT CAST(B.ImplementationDate AS DATE)) AS CountofImplementationsPerAccount
                    FROM dexposure A
                    LEFT JOIN `{env_prefix}-cdh`.primary.settlementexposure B
                    ON regexp_replace(A.ExposureKey, '[^0-9]', '') = regexp_replace(B.ExposureKey, '[^0-9]', '')
                    AND regexp_replace(A.CDI, '[^0-9]', '') = regexp_replace(B.CDI, '[^0-9]', '')
                    AND A.PortfolioName = B.PortfolioName
                    WHERE CAST(B.ImplementationDate AS DATE) IS NOT NULL
                    AND CAST(B.ImplementationDate AS DATE) > '{cutoffdate}'
                    AND CAST(B.ImplementationDate AS DATE) < '{LastWorkingDate}'
                    GROUP BY A.ExposureKey
""")

impl_count.createOrReplaceTempView("impl_count")

In [0]:
# %sql
# SELECT *
# FROM impl_date
# WHERE ExposureKey = 'MRC989002937097507'

### Application Date

In [0]:
sttlexp_appdate = spark.sql(f"""
                    SELECT CDI, ExposureKey, MAX(ApplicationDate) AS ApplicationDate
                    FROM
                    (
                    SELECT DISTINCT A.CDI, A.ExposureKey, CASE WHEN CAST(B.SubmissionDate AS DATE) > CAST(B.ImplementationDate AS DATE) THEN CAST(B.SubmissionDate AS DATE) ELSE CAST(B.ImplementationDate AS DATE) END AS ApplicationDate
                    FROM dexposure A
                    LEFT JOIN settlementexposure B
                    ON regexp_replace(A.ExposureKey, '[^0-9]', '') = regexp_replace(B.ExposureKey, '[^0-9]', '')
                    AND regexp_replace(A.CDI, '[^0-9]', '') = regexp_replace(B.CDI, '[^0-9]', '')
                    AND A.PortfolioName = B.PortfolioName
                    )
                    GROUP BY CDI, ExposureKey
""")

sttlexp_appdate.createOrReplaceTempView("sttlexp_appdate")

app_tmp = spark.sql(f"""
                    SELECT A.CDI, A.ExposureKey, MAX(CAST(B.CreatedDate AS DATE)) AS CreatedDate
                    FROM dexposure A
                    LEFT JOIN `{env_prefix}-cdh`.primary.application B
                    ON regexp_replace(A.CDI, '[^0-9]', '') = regexp_replace(B.CDI, '[^0-9]', '')
                    AND A.PortfolioName = B.PortfolioName
                    WHERE B.BusinessDate = {LastWorkingDate!r}
                    GROUP BY A.CDI, A.ExposureKey
""")

app_tmp.createOrReplaceTempView("app_tmp")

app_date = spark.sql(f"""
    SELECT 
        A.ExposureKey,
        MAX(
            CASE 
                WHEN CreatedDate IS NULL THEN ApplicationDate
                WHEN ApplicationDate IS NULL THEN CreatedDate
                WHEN ABS(DATEDIFF(ApplicationDate, CreatedDate)) < 7 THEN 
                    CASE 
                        WHEN ApplicationDate < CreatedDate THEN ApplicationDate 
                        ELSE CreatedDate 
                    END
                ELSE 
                    CASE 
                        WHEN ApplicationDate > CreatedDate THEN ApplicationDate 
                        ELSE CreatedDate 
                    END
            END
        ) AS AppliedDate
    FROM sttlexp_appdate A
    LEFT JOIN app_tmp B
        ON A.CDI = B.CDI
       AND A.ExposureKey = B.ExposureKey
    GROUP BY A.ExposureKey
""")

app_date.createOrReplaceTempView("app_date")

In [0]:
# %sql
# SELECT *
# FROM app_date
# WHERE ExposureKey = 'MRC989002937097507'

### Settlement Amount

In [0]:
settlementamount = spark.sql(f"""
                    SELECT ExposureKey, SUM(SettlementAmount) AS SettlementAmount
                    FROM settlementexposure
                    GROUP BY ExposureKey
""")

settlementamount.createOrReplaceTempView("settlementamount")

dexposure_sttl = spark.sql(f"""
                    SELECT DISTINCT A.ExposureKey, A.InitialAssignmentAmount
                    FROM dexposure A
                    INNER JOIN drestructuring B
                    ON A.ExposureKey = B.NewExposureKey
                    WHERE A.ExposureStatus <> 'Writen Off'
""")

dexposure_sttl.createOrReplaceTempView("dexposure_sttl")

sttl_amount = spark.sql(f"""
                    SELECT DISTINCT A.ExposureKey,
                            CASE WHEN C.ExposureKey IS NOT NULL THEN C.InitialAssignmentAmount ELSE B.SettlementAmount END AS SettlementAmount,
                            CASE WHEN C.ExposureKey IS NOT NULL THEN 'Restructuring' ELSE 'Settlement' END AS SettlementRestructuringStatus
                    FROM dexposure A
                    LEFT JOIN settlementamount B
                    ON A.ExposureKey = B.ExposureKey
                    LEFT JOIN dexposure_sttl C
                    ON A.ExposureKey = C.ExposureKey
""")

sttl_amount.createOrReplaceTempView("sttl_amount")

In [0]:
# %sql
# SELECT *
# FROM sttl_amount
# WHERE ExposureKey = 'MRC989008494512202'

### Haircut

In [0]:
restructuring_latestdate = spark.sql(f"""
                                SELECT A.ExposureKey, MAX(B.BusinessDate) AS LatestBusinessDate
                                FROM dexposure_sttl A
                                INNER JOIN `{env_prefix}-cdh`.primary.restructuring B
                                ON A.ExposureKey = B.NewExposureKey
                                WHERE B.SourceSystem = 'profits'
                                AND B.BusinessDate <= {LastWorkingDate!r} AND RestructuringDiscountAmount IS NOT NULL
                                GROUP BY A.ExposureKey
""")

restructuring_latestdate.createOrReplaceTempView("restructuring_latestdate")

restructuring_atlatestdate = spark.sql(f"""
                                SELECT DISTINCT A.ExposureKey,
                                        B.RestructuringDiscountAmount, CAST(B.RestructuringDiscountAmount AS FLOAT)/CAST(B.RestructuringAmount AS FLOAT) AS HaircutPercentage,
                                        CAST(B.RestructuringAmount AS FLOAT) - CAST(B.RestructuringDiscountAmount AS FLOAT) AS TotalAmountDueAfterHaircut
                                FROM restructuring_latestdate A
                                INNER JOIN `{env_prefix}-cdh`.primary.restructuring B
                                ON A.ExposureKey = B.NewExposureKey
                                AND A.LatestBusinessDate = B.BusinessDate
                                WHERE B.SourceSystem = 'profits'
""")

restructuring_atlatestdate.createOrReplaceTempView("restructuring_atlatestdate")

In [0]:
haircut = spark.sql(f"""
                SELECT DISTINCT A.ExposureKey,
                        CASE WHEN A.SettlementRestructuringStatus = 'Restructuring' THEN CAST(C.RestructuringDiscountAmount AS FLOAT) ELSE B.DiscountAmount END AS HaircutAmount,
                        CASE WHEN A.SettlementRestructuringStatus = 'Restructuring' THEN C.HaircutPercentage
                        ELSE 1 - (A.SettlementAmount/B.ProposedAmount) END AS HaircutPercentage,
                        1 - (A.SettlementAmount/B.PrincipalProposedAmount) AS HaircutPrincipalPercentage,
                        CASE WHEN A.SettlementRestructuringStatus = 'Restructuring' THEN C.TotalAmountDueAfterHaircut ELSE B.RemainAmount END AS TotalAmountDueAfterHaircut
                FROM sttl_amount A
                LEFT JOIN settlementexposure B
                ON A.ExposureKey = B.ExposureKey
                LEFT JOIN restructuring_atlatestdate C
                ON A.ExposureKey = C.ExposureKey
""")

haircut.createOrReplaceTempView("haircut")

In [0]:
# %sql
# SELECT *
# FROM haircut
# WHERE ExposureKey = 'MRC40000078452'

### Misc. Settlement fields

In [0]:
settlementtype = spark.sql(f"""
                    SELECT *,
                    CASE WHEN InLawFlag = 1 THEN UnderLawType ELSE SettlementType END AS ApplicationType
                    FROM
                    (
                    SELECT DISTINCT
                        A.ExposureKey, A.CDI,
                        B.InLawFlag, B.UnderLawType,
                        CASE
                        WHEN SettlementType IN ('AB Migration') THEN 'Migration Settlement'
                        WHEN SettlementType IN ('CELL Ν3869 Διατραπεζική Επικοινωνία', 'L3869 AB Migration', 'L3869 Court Decision') THEN 'Migration L3869'
                        WHEN SettlementType IN ('L3869 Temporary', 'L3869_Temporary') THEN 'Migration L3869 Temporary'
                        WHEN SettlementType IN ('L3869_Approved', 'Mercury_N.3869 Οριστική Απόφαση', 'MRC Ν3869 Οριστική', 'MRC Ν3869 Οριστική (pending to Conf.Total amount)', 'Νόμος 3869 – Δικαστική Απόφαση') THEN 'L3869 Approved'
                        WHEN SettlementType IN ('L4605_Approved') THEN 'Migration L4605 Approved'
                        WHEN SettlementType IN ('L4605_Temporary') THEN 'Migration L4605 Temporary'
                        WHEN SettlementType IN ('MRC Ν3869 Προσωρινή', 'MRC Ν3869 Προσωρινή (pending to Conf.Total amount)') THEN 'L3869 Pending'
                        WHEN SettlementType IN ('MRC Ν3869 Expired') THEN 'L3869 with No Active Plan'
                        WHEN SettlementType IN ('DPO within Limits') THEN
                                CASE WHEN C.SubmissionDate > '{migrationDate}' THEN 'Settlement' ELSE 'Migration' END
                        WHEN SettlementType IN ('L.4738 OCW with HAIRCUT _ All Rels') THEN 'OCW Approved'
                        WHEN SettlementType IN ('Mercury Letter Offer') THEN 'Interim Period Settlement'
                        WHEN SettlementType IN (
                                'Mercury Out of Mandate', 'Mercury Out of Mandate-AO Mass Approved', 'Mercury Preapproved as Declared', 
                                'Mercury Preapproved with Documents', 'Mercury Preapproved with Interview', 'Mercury Reignition L3869', 
                                'Out of Mandate', 'Out_of_Mandate', 'Preapproved_as_Declared', 'Preapproved_with_Documents', 
                                'Preapproved_with_Interview', 'Settlement Manual Correction', 'Venus Preapproved with Documents', 
                                'Venus within Limits'
                        ) THEN 'Cepal Settlement'
                        WHEN SettlementType IN ('Άρση Εγγυητικής Σχέσης') THEN 'Cepal Settlement on Guarantor'
                        ELSE NULL
                        END AS SettlementType
                    FROM dexposure A
                    LEFT JOIN dunderlaw_active B
                    ON A.CDI = B.CDI
                    LEFT JOIN settlementexposure C
                    ON regexp_replace(A.ExposureKey, '[^0-9]', '') = regexp_replace(C.ExposureKey, '[^0-9]', '')
                    AND regexp_replace(A.CDI, '[^0-9]', '') = regexp_replace(C.CDI, '[^0-9]', '')
                    AND A.PortfolioName = C.PortfolioName
                    )
""")

settlementtype.createOrReplaceTempView("settlementtype")

In [0]:
settlementstatus = spark.sql(f"""
                    SELECT DISTINCT
                        A.ExposureKey, A.CDI,
                        CASE WHEN B.SettlementStatus IS NOT NULL THEN B.SettlementStatus
                        WHEN C.RestructuringType IS NOT NULL THEN 'Restructure' ELSE 'No Settlement/Restructure' END AS Status,
                        CASE WHEN A.HasActiveSettlementFlag IS TRUE AND B.IsMigratedFlag IS TRUE THEN 'Yes' ELSE 'No' END AS SettlementFromMigration
                    FROM dexposure A
                    LEFT JOIN settlementexposure B
                    ON regexp_replace(A.ExposureKey, '[^0-9]', '') = regexp_replace(B.ExposureKey, '[^0-9]', '')
                    AND regexp_replace(A.CDI, '[^0-9]', '') = regexp_replace(B.CDI, '[^0-9]', '')
                    AND A.PortfolioName = B.PortfolioName
                    LEFT JOIN drestructuring C
                    ON A.ExposureKey = C.NewExposureKey
""")

settlementstatus.createOrReplaceTempView("settlementstatus")

### Installments

In [0]:
overdueinstallments = spark.sql(f"""
                    SELECT A.ExposureKey, COUNT(B.InstallmentNumber) AS OverdueInstallments
                    FROM dexposure A
                    LEFT JOIN `{env_prefix}-cdh`.derivative.dpaymentschedule B
                    ON regexp_replace(A.ExposureKey, '[^0-9]', '') = regexp_replace(B.ExposureKey, '[^0-9]', '')
                    AND A.PortfolioName = B.PortfolioName
                    WHERE B.BusinessDate = {LastWorkingDate!r}
                    AND B.InstallmentStatus = 'Overdue'
                    GROUP BY A.ExposureKey
""")

overdueinstallments.createOrReplaceTempView("overdueinstallments")

monthlyinstallmentamount = spark.sql(f"""
                    SELECT A.ExposureKey, SUM(-B.InstallmentAmount) AS MonthlyInstallmentAmount
                    FROM dexposure A
                    LEFT JOIN `{env_prefix}-cdh`.derivative.dpaymentschedule B
                    ON regexp_replace(A.ExposureKey, '[^0-9]', '') = regexp_replace(B.ExposureKey, '[^0-9]', '')
                    AND A.PortfolioName = B.PortfolioName
                    WHERE B.BusinessDate = {LastWorkingDate!r}
                    AND B.InstallmentDate >= {FirstWorkingDate!r} AND B.InstallmentDate <= {LastWorkingDate!r}
                    GROUP BY A.ExposureKey
""")

monthlyinstallmentamount.createOrReplaceTempView("monthlyinstallmentamount")

installments = spark.sql(f"""
                    SELECT
                      A.ExposureKey,
                      MAX(B.NumOfinstalments)            AS NumOfinstalments,          -- or another agg
                      MAX(B.PastInstalmentNumber)        AS PastInstalmentNumber,
                      MAX(C.OverdueInstallments)         AS OverdueInstallments,
                      CASE
                        WHEN MAX(CASE WHEN B.NumOfinstalments <= 6 THEN 1 ELSE 0 END) = 1
                          THEN 'Short Term (6M)'
                        ELSE 'Long Term (+6M)'
                      END AS ShortTerm_LongTermFlag,
                      MAX(B.FirstInstalmentDate)         AS FirstInstalmentDate,        -- pick desired agg
                      MAX(D.MonthlyInstallmentAmount)    AS MonthlyInstallmentAmount
                    FROM dexposure A
                    LEFT JOIN settlementexposure B
                      ON regexp_replace(A.ExposureKey, '[^0-9]', '') = regexp_replace(B.ExposureKey, '[^0-9]', '')
                     AND regexp_replace(A.CDI, '[^0-9]', '') = regexp_replace(B.CDI,  '[^0-9]', '')
                     AND A.PortfolioName = B.PortfolioName
                    LEFT JOIN overdueinstallments C
                      ON A.ExposureKey = C.ExposureKey
                    LEFT JOIN monthlyinstallmentamount D
                      ON A.ExposureKey = D.ExposureKey
                    GROUP BY A.ExposureKey;
""")

installments.createOrReplaceTempView("installments")

# we use column FirstInstalmentDate from settlementexposure
firstinstallmentdate = spark.sql(f"""
                    SELECT A.ExposureKey, MIN(CAST(B.InstallmentDate AS DATE)) AS FirstInstallmentDate
                    FROM dexposure A
                    LEFT JOIN `{env_prefix}-cdh`.derivative.dpaymentschedule B
                    ON regexp_replace(A.ExposureKey, '[^0-9]', '') = regexp_replace(B.ExposureKey, '[^0-9]', '')
                    AND A.PortfolioName = B.PortfolioName
                    WHERE B.InstallmentNumber = 1
                    GROUP BY A.ExposureKey
""")

firstinstallmentdate.createOrReplaceTempView("firstinstallmentdate")

# we use column PastInstalmentNumber from settlementexposure
pastinstallments = spark.sql(f"""
                    SELECT A.ExposureKey, COUNT(B.InstallmentNumber) AS PastInstallments
                    FROM dexposure A
                    LEFT JOIN `{env_prefix}-cdh`.derivative.dpaymentschedule B
                    ON regexp_replace(A.ExposureKey, '[^0-9]', '') = regexp_replace(B.ExposureKey, '[^0-9]', '')
                    AND A.PortfolioName = B.PortfolioName
                    WHERE B.BusinessDate = {LastWorkingDate!r}
                    AND CAST(B.InstallmentDate AS DATE) < {LastWorkingDate!r}
                    GROUP BY A.ExposureKey
""")

pastinstallments.createOrReplaceTempView("pastinstallments")

### Final Settlement Table

In [0]:
dexposure_dueamount = spark.sql(f"""
                    SELECT DISTINCT A.ExposureKey, A.DueAmount, B.KeptPercentage,A.CDI
                    FROM dexposure A
                    LEFT JOIN settlementexposure B
                    ON regexp_replace(A.ExposureKey, '[^0-9]', '') = regexp_replace(B.ExposureKey, '[^0-9]', '')
                    AND regexp_replace(A.CDI, '[^0-9]', '') = regexp_replace(B.CDI, '[^0-9]', '')
                    AND A.PortfolioName = B.PortfolioName
""")

dexposure_dueamount.createOrReplaceTempView("dexposure_dueamount")

settlement_fields = spark.sql(f"""
                    SELECT DISTINCT A.ExposureKey, A.DueAmount,A.CDI,
                            A.KeptPercentage AS KeptRate,
                            B.ImplementedDate,
                            C.CountofImplementationsPerAccount,
                            D.AppliedDate,
                            E.SettlementAmount,
                            F.HaircutAmount, F.HaircutPercentage, F.HaircutPrincipalPercentage, F.TotalAmountDueAfterHaircut,
                            G.SettlementType, G.ApplicationType,
                            H.Status, H.SettlementFromMigration,
                            I.NumOfinstalments AS NumberOfInstallments, I.PastInstalmentNumber AS NumberOfPastInstallments, I.OverdueInstallments AS NumberOfMissedInstallments, I.ShortTerm_LongTermFlag, I.FirstInstalmentDate AS FirstInstallmentDate, I.MonthlyInstallmentAmount
                    FROM dexposure_dueamount A
                    LEFT JOIN impl_date B
                    ON A.ExposureKey = B.ExposureKey
                    LEFT JOIN impl_count C
                    ON A.ExposureKey = C.ExposureKey
                    LEFT JOIN app_date D
                    ON A.ExposureKey = D.ExposureKey
                    LEFT JOIN sttl_amount E
                    ON A.ExposureKey = E.ExposureKey
                    LEFT JOIN haircut F
                    ON A.ExposureKey = F.ExposureKey
                    LEFT JOIN settlementtype G
                    ON A.ExposureKey = G.ExposureKey and A.CDI = G.CDI
                    LEFT JOIN settlementstatus H
                    ON A.ExposureKey = H.ExposureKey
                    LEFT JOIN installments I
                    ON A.ExposureKey = I.ExposureKey
""")

settlement_fields.createOrReplaceTempView("settlement_fields")

In [0]:
#%sql
#SELECT *
#FROM settlement_fields
#WHERE ExposureKey = 'MRC989002937097507'

In [0]:
# %sql
# SELECT *
# FROM installments
# WHERE ExposureKey = 'MRC989008494512202'

In [0]:
# %sql
# select *
# FROM settlementtype
# where ExposureKey = 'MRC4509034583045007'

In [0]:
# %sql
# select *
# FROM settlementstatus
# where ExposureKey = 'MRC4509030102445009'

In [0]:
# %sql
# SELECT COUNT(*)
# FROM installments

## Collections

In [0]:
MonthlyPayments = spark.sql(f"""
                    SELECT PortfolioName, SourceExposureKey as ExposureKey, SUM(Amount) AS MonthlyPayments
                    FROM `{env_prefix}-cdh`.derivative.dcollection
                    WHERE PortfolioName = {portfolioName!r}
                    AND ActualDate BETWEEN '{FirstWorkingDate}' AND '{LastWorkingDate}' AND BusinessDate = '{LastWorkingDate}'
                    AND PaymentTypeGroup = 'Cash'
                    GROUP BY PortfolioName, SourceExposureKey
""")

MonthlyPayments.createOrReplaceTempView("MonthlyPayments")

TotalPaymentsAfterLTSA = spark.sql(f"""
                    SELECT PortfolioName, SourceExposureKey as ExposureKey, SUM(Amount) AS TotalPaymentsAfterLTSA
                    FROM `{env_prefix}-cdh`.derivative.dcollection
                    WHERE PortfolioName = {portfolioName!r}
                    AND CASE WHEN PortfolioName = 'Mercury' THEN ActualDate BETWEEN '{cutoffdate}' AND '{LastWorkingDate}' ELSE ActualDate < '{LastWorkingDate}' END
                    AND BusinessDate = '{LastWorkingDate}'
                    AND PaymentTypeGroup = 'Cash'
                    GROUP BY PortfolioName, SourceExposureKey
""")

TotalPaymentsAfterLTSA.createOrReplaceTempView("TotalPaymentsAfterLTSA")

LastPaymentDate = spark.sql(f"""
                    SELECT PortfolioName, SourceExposureKey as ExposureKey, MAX(ActualDate) AS LastPaymentDate
                    FROM `{env_prefix}-cdh`.derivative.dcollection
                    WHERE PortfolioName = {portfolioName!r}
                    AND BusinessDate = '{LastWorkingDate}'
                    AND PaymentTypeGroup = 'Cash'
                    GROUP BY PortfolioName, SourceExposureKey
""")

LastPaymentDate.createOrReplaceTempView("LastPaymentDate")

Collections = spark.sql(f"""
                    SELECT DISTINCT A.PortfolioName, A.ExposureKey, MonthlyPayments, TotalPaymentsAfterLTSA, LastPaymentDate
                    FROM MonthlyPayments A
                    LEFT JOIN TotalPaymentsAfterLTSA B
                    ON A.PortfolioName = B.PortfolioName AND A.ExposureKey = B.ExposureKey
                    LEFT JOIN LastPaymentDate C
                    ON A.PortfolioName = C.PortfolioName AND A.ExposureKey = C.ExposureKey
""")

Collections.createOrReplaceTempView("Collections")

## Assignment

In [0]:
assignment = spark.sql(f"""
                    SELECT DISTINCT PortfolioName, ExposureKey,
                            case when Channel = 'External LF' and AssignmentProcess in ('Collection', 'Collection_Judicial_Escalation') then concat(ExternalCompanyFullName, ' Collection')
                            when Channel = 'External LF' and AssignmentProcess in ('Lf_Offer') then concat(ExternalCompanyFullName, ' Offer')
                            when Channel = 'External LF' and AssignmentProcess not in ('Collection', 'Collection_Judicial_Escalation', 'Lf_Offer') then concat(ExternalCompanyFullName, ' Other')
                            else Channel end as AssignedChannel
                            from `{env_prefix}-cdh`.primary.assignment
                            WHERE PortfolioName = {portfolioName!r} AND BusinessDate = '{LastWorkingDate}'
""")

assignment.createOrReplaceTempView("assignment")

## L3869 fields

In [0]:
l3869submissiondate = spark.sql(f"""
                    SELECT A.CDI, MAX(SubmissionDate) AS L3869SubmissionDate
                    FROM `p-cdh`.derivative.dunderlaw A
                    INNER JOIN dparty B
                        ON A.CDI = B.CDI
                    WHERE A.PortfolioName = 'Mercury'
                        AND A.UnderLawType = 'N3869'
                    AND EXISTS (
                                SELECT 1
                                FROM dexposure C
                                WHERE C.CDI = A.CDI
                                  AND C.PortfolioName = 'Mercury'
                    )
                    GROUP BY A.CDI
""")

l3869submissiondate.createOrReplaceTempView("l3869submissiondate")

applicationstatusdate = spark.sql(f"""
                    SELECT DISTINCT A.CDI, ApplicationStatusDate
                    FROM `{env_prefix}-cdh`.derivative.dunderlaw A
                    INNER JOIN dparty B
                    ON A.CDI = B.CDI
                    WHERE PortfolioName = {portfolioName!r}
                    AND BusinessDate = {LastWorkingDate!r}
                    AND UnderLawType = 'N3869'
                    AND LawStatus in ('Εκτός Ν3869', 'Εκτός Ν3869 - Απορριπτική Δικαστ. Απόφαση', 'Εκτός Ν3869 - Παραίτηση', 'Εκτός ν3869 - Ματαίωση')
""")

applicationstatusdate.createOrReplaceTempView("applicationstatusdate")

prvapplicationstatusdate = spark.sql(f"""
                    SELECT A.CDI, MAX(BusinessDate) AS LatestBusinessDate
                    FROM `{env_prefix}-cdh`.derivative.dparty A
                    INNER JOIN applicationstatusdate B
                    ON A.CDI = B.CDI
                    WHERE A.PortfolioName = {portfolioName!r}
                    AND BusinessDate < ApplicationStatusDate
                    GROUP BY A.CDI
""")

prvapplicationstatusdate.createOrReplaceTempView("prvapplicationstatusdate")

prvsegmentation = spark.sql(f"""
                    SELECT DISTINCT A.CDI, Segment AS PreviousSegmentation
                    FROM `{env_prefix}-cdh`.derivative.dparty A
                    INNER JOIN prvapplicationstatusdate B
                    ON A.CDI = B.CDI
                    AND A.BusinessDate = B.LatestBusinessDate
                    WHERE A.PortfolioName = {portfolioName!r}
""")

prvsegmentation.createOrReplaceTempView("prvsegmentation")

l3869totaldebt = spark.sql(f"""
                    SELECT DISTINCT CDI, ExposureKey,
                                CASE WHEN ExposureStatus NOT IN ('Define Delay', 'Writen Off')
                                THEN DuePrincipal +
                                DueInterest +
                                PenaltyInterest +
                                DueExpense +
                                DueUnrealizedInterest +
                                UnrealizedInterest +
                                PenaltyUnrealizedInterest
                                ELSE OutstandingTotalBalance END AS L3869TotalDebt
                    FROM dexposure
""")

l3869totaldebt.createOrReplaceTempView("l3869totaldebt")

dunderlaw = spark.sql(f"""
                    SELECT DISTINCT B.CDI, L3869Status,
                            CASE WHEN UnderLawType = 'N3869' THEN LawStatus ELSE NULL END AS L3869Status1,
                            CASE WHEN UnderLawType = 'N3869' AND LawStatus IN ('Εκτός Ν3869', 'Εκτός Ν3869 - Απορριπτική Δικαστ. Απόφαση',
                            'Εκτός Ν3869 - Παραίτηση', 'Εκτός ν3869 - Ματαίωση') THEN date_format(ApplicationStatusDate, 'MMMM-yyyy') ELSE NULL END AS L3869MonthExit,
                            CASE WHEN UnderLawType = 'N3869' THEN CourtOutcome ELSE NULL END AS L3869CourtDecision,
                            CASE WHEN UnderLawType = 'N3869' AND LawStatus IS NOT NULL AND CourtOutcome IS NOT NULL THEN CourtDate ELSE NULL END AS L3869FinalCourtDate,
                            D.PreviousSegmentation,
                            C.L3869SubmissionDate,
                            CASE WHEN UnderLawType = 'N3869' AND LawStatus = 'Εκτός Ν3869 - Παραίτηση' THEN ApplicationStatusDate ELSE NULL END AS L3869ResignationDate,
                            CASE WHEN ApplicationStatus IN ('Rejected', 'Resigned') AND UnderLawType = 'N3869' THEN 'Yes' ELSE 'No' END AS EkptosiFlag,
                            CASE WHEN ApplicationStatus IN ('Rejected', 'Resigned') AND UnderLawType = 'N3869' THEN LawStatus ELSE NULL END AS EkptosiStatus,
                            E.Total_Balance_ML AS `L3869MLΑmount`,
                            CASE WHEN E.Has_ML IS NOT NULL THEN E.Has_ML ELSE 'No' END AS MLinABExist
                    FROM `{env_prefix}-cdh`.derivative.dunderlaw A
                    INNER JOIN dparty B
                    ON A.CDI = B.CDI
                    LEFT JOIN l3869submissiondate C
                    ON B.CDI = C.CDI
                    LEFT JOIN prvsegmentation D
                    ON B.CDI = D.CDI
                    LEFT JOIN `qa-reportinputfiles`.investor.mlexist E
                    ON regexp_replace(A.CDI, '[^0-9]', '') = regexp_replace(E.CIF, '[^0-9]', '')
                    AND A.PortfolioName = E.Portfolio
                    WHERE A.PortfolioName = {portfolioName!r}
                    AND BusinessDate= {LastWorkingDate!r}
                     """)

dunderlaw.createOrReplaceTempView("dunderlaw")

In [0]:
# %sql
# SELECT *
# FROM dunderlaw
# WHERE CDI IN ('MRC0001894673', 'MRC0009908632')

## Letter Flag

In [0]:
letterflagexist = spark.sql(f"""
                    SELECT DISTINCT A.CDI AS LetterFlagCDI
                    FROM dexposure A
                    INNER JOIN LetterFlag B
                    ON A.PartyTaxRegistrationNumber = B.AFM
""")

letterflagexist.createOrReplaceTempView("letterflagexist")

### Campaign Name

In [0]:
campaignname = spark.sql(f"""
                    select ca.accountnumber, ca.campaignname
                    from `{env_prefix}-cdh`.reference.mercury_campaign_activity ca 
                    where ca.ActiveFlagOnAccountLevel = 1
                    and ca.ClosingDateOnAccountLevel = '9999-12-31' --this filter is used to deduplicate accounts that have closed campaigns
""")
 
campaignname.createOrReplaceTempView("campaignname")

## Contact fields

In [0]:
# @IntergationDate = '2018-12-24'
# there was a last part to this query BusinessDate >= @IntergationDate

# @IntergationDate = '2018-12-24'
# there was a last part to this query BusinessDate >= @IntergationDate

eap = spark.sql(f"""
WITH raw_eap AS (
  -- interactions (qc)
  SELECT 
      intr.ExposureKey,
      intr.InteractionDate      AS Last_EAP_Date,
      intr.InteractionTypeCode  AS ActivityTypeKey,
      CAST(NULL AS STRING)      AS CompletionResult,
      intr.SourceSystem         AS SourceSystem
  FROM `{env_prefix}-cdh`.primary.interaction intr
  WHERE intr.InteractionTypeCode IN ('5196','5199','5200','5201','6542')
    AND intr.BusinessDate = {LastWorkingDate!r}
    AND intr.SourceSystem = 'qc'

  UNION ALL

  -- legalaction (qc, by ExposureKey)
  SELECT 
      la.ExposureKey,
      la.LegalActionDate        AS Last_EAP_Date,
      la.LegalActionTypeCode    AS ActivityTypeKey,
      CAST(NULL AS STRING)      AS CompletionResult,
      la.SourceSystem           AS SourceSystem
  FROM `{env_prefix}-cdh`.primary.legalaction la
  WHERE la.LegalActionTypeCode IN ('5196','5199','5200','5201','6542')
    AND la.LegalActionDate <= {LastWorkingDate!r}
    AND la.SourceSystem = 'qc'

  UNION ALL

  -- legalaction (arotron, via CDI)
  SELECT 
      exp.ExposureKey,
      la.LegalActionDate        AS Last_EAP_Date,
      la.LegalActionTypeCode    AS ActivityTypeKey,
      la.CompletionResult       AS CompletionResult,
      la.SourceSystem           AS SourceSystem
  FROM `{env_prefix}-cdh`.primary.legalaction la
  INNER JOIN `{env_prefix}-cdh`.primary.exposure exp
    ON la.CDI = exp.CDI
   AND exp.BusinessDate = la.BusinessDate
  WHERE la.LegalActionType = 'Έρευνα Ακίνητης Περιουσίας'
    AND la.LegalActionDate <= {LastWorkingDate!r}
    AND la.BusinessDate = {LastWorkingDate!r}
    AND la.SourceSystem = 'arotron'
),

EAP_CTE AS (
  SELECT
      ExposureKey,
      Last_EAP_Date,
      ActivityTypeKey,
      CompletionResult,
      SourceSystem,
      ROW_NUMBER() OVER (
          PARTITION BY ExposureKey
          ORDER BY Last_EAP_Date DESC
      ) AS RowNumber
  FROM raw_eap
)

SELECT DISTINCT
    dexp.ExposureKey,
    EAP_CTE.Last_EAP_Date,
    CASE 
      WHEN (EAP_CTE.SourceSystem = 'qc'
            AND EAP_CTE.ActivityTypeKey IN ('5199','5200')) THEN 'Completed'
      WHEN (EAP_CTE.SourceSystem = 'arotron'
            AND EAP_CTE.CompletionResult IS NOT NULL) THEN 'Completed'
      ELSE 'In Progress'
    END AS EAP_Status
FROM dexposure dexp
LEFT JOIN EAP_CTE
  ON dexp.ExposureKey = EAP_CTE.ExposureKey
WHERE EAP_CTE.RowNumber = 1
ORDER BY EAP_CTE.Last_EAP_Date DESC
""")

eap.createOrReplaceTempView("eap")


In [0]:
# @IntergationDate = '2018-12-24'
# there was a last part to this query BusinessDate >= @IntergationDate

re = spark.sql(f"""
    SELECT DISTINCT
        dexp.ExposureKey,
        CASE WHEN re.ExposureKey IS NOT NULL THEN 'Yes' ELSE 'No' END AS Positive_RE_Exists
    FROM dexposure dexp 
    LEFT JOIN `{env_prefix}-cdh`.primary.interaction re
        ON dexp.ExposureKey = re.ExposureKey
            AND re.InteractionTypeCode = '5200'
            AND re.BusinessDate = {LastWorkingDate!r}
            AND re.SourceSystem = 'qc'
""")

re.createOrReplaceTempView("re")

In [0]:
# When PortfolioName = 'Mercury' and interactionDate > '2019-07-01'
# When PortfolioName in ('Orbit', 'Light') and interactionDate >= '2022-03-28'
# When PortfolioName = 'Cell' and interactionDate >= '2023-10-23'
# When PortfolioName = 'Pearl' and interactionDate >= '2024-02-19'

#This is how rejected offers were implemented on premise, BRD spec updated and generated the next calc. 
re_off_ba = spark.sql(f"""
                        WITH REJECTED_OFFER AS (
                            SELECT 
                                intr.ExposureKey,
                                MIN(intr.InteractionDate) AS DateOfFirstRejectedOffer,
                                MAX(intr.InteractionDate) AS DateOfLastRejectedOffer,
                                COUNT(intr.InteractionKey) AS TotalCountOfRejectedOffers
                            FROM `{env_prefix}-cdh`.primary.interaction intr 
                            WHERE InteractionTypeCode IN ('3009', '2017', '2018', '3046', '3007', '3008', '2010', '2011', '2016', '2013', '2014', '2015','2019')  
                                AND intr.BusinessDate = {LastWorkingDate!r}
                                AND intr.InteractionDate >= {cutoffdate!r}
                            GROUP BY intr.ExposureKey

                       )
    SELECT 
        dexp.ExposureKey,
        COALESCE(rejoffer.TotalCountOfRejectedOffers, 0) AS TotalCountOfRejectedOffers
    FROM dexposure dexp 
    LEFT JOIN REJECTED_OFFER rejoffer
        ON dexp.ExposureKey = rejoffer.ExposureKey
""")

re_off_ba.createOrReplaceTempView("re_off_ba")

In [0]:
# When PortfolioName = 'Mercury' and interactionDate > '2019-07-01'
# When PortfolioName in ('Orbit', 'Light') and interactionDate >= '2022-03-28'
# When PortfolioName = 'Cell' and interactionDate >= '2023-10-23'
# When PortfolioName = 'Pearl' and interactionDate >= '2024-02-19'

#AttemptOutcome is null in many records. 
neg_attempts = spark.sql(f"""
                        WITH NEGATIVE_ATTEMPTS AS (
                            SELECT 
                                intr.ExposureKey,
                                MIN(intr.InteractionDate) AS DateOfFirstNegativeAttempt,
                                MAX(intr.InteractionDate) AS DateOfLastNegativeAttempt,
                                COUNT(intr.InteractionKey) AS TotalCountOfNegativeAttempts
                            FROM `{env_prefix}-cdh`.primary.interaction intr 
                            WHERE AttemptOutcome = 'negative'
                                AND intr.BusinessDate = {LastWorkingDate!r}
                                AND intr.InteractionDate >= {cutoffdate!r}
                                AND intr.SourceSystem = 'qc'
                            GROUP BY intr.ExposureKey

                       )
    SELECT 
        dexp.ExposureKey,
        COALESCE(neg_attempts.TotalCountOfNegativeAttempts, 0) AS TotalCountOfNegativeAttempts, 
        neg_attempts.DateOfFirstNegativeAttempt, 
        neg_attempts.DateOfLastNegativeAttempt
    FROM dexposure dexp 
    LEFT JOIN NEGATIVE_ATTEMPTS neg_attempts
        ON dexp.ExposureKey = neg_attempts.ExposureKey
""")

neg_attempts.createOrReplaceTempView("neg_attempts")

In [0]:
#AttemptOutcome is null in many records. 
neg_reason = spark.sql(f"""
                        SELECT DISTINCT
                            intr.ExposureKey,
                            intr.InteractionType, 
                            intr.InteractionComments
                        FROM `{env_prefix}-cdh`.primary.interaction intr 
                        INNER JOIN neg_attempts
                            ON neg_attempts.ExposureKey = intr.ExposureKey
                        WHERE AttemptOutcome = 'negative'
                            AND intr.BusinessDate = {LastWorkingDate!r}
                            AND intr.InteractionDate = neg_attempts.DateOfFirstNegativeAttempt
                            AND intr.ContactType IN ('Right Person Contact', '3rd Person Contact','RIGHT PARTY CONTACT')
                            AND intr.SourceSystem = 'qc'
""")

neg_reason.createOrReplaceTempView("neg_reason")

In [0]:
attempt_dates_after_LTSA = spark.sql(f"""
                                        WITH ATTEMPT_DATES_AFTER_LTSA AS (
                                            SELECT 
                                                intr.CDI,
                                                MIN(intr.InteractionDate) AS DateOfFirstAttempt,
                                                MAX(intr.InteractionDate) AS DateOfLastAttempt,
                                                COUNT(intr.InteractionKey) AS TotalCountOfAttempts
                                            FROM `{env_prefix}-cdh`.primary.interaction intr 
                                            WHERE   intr.BusinessDate in ({LastWorkingDate!r},'2023-12-29')
                                                AND intr.InteractionDate >= {cutoffdate!r}
                                                AND intr.SourceSystem = 'qc'
                                                AND intr.ContactType IN ('Right Person Contact', '3rd Person Contact', 'No Contact','RIGHT PARTY CONTACT')
                                            GROUP BY intr.CDI
                                            UNION 
                                            SELECT
                                                di.CDI, 
                                                MIN(CAST(di.DateHour AS DATE)) AS DateOfFirstAttempt,
                                                MAX(CAST(di.DateHour AS DATE)) AS DateOfLastAttempt,
                                                COUNT(di.DialerKey) AS TotalCountOfAttempts
                                            FROM `{env_prefix}-cdh`.primary.dialer di
                                            WHERE di.BusinessDate = {LastWorkingDate!r}
                                                AND di.CallDisposition IS NOT NULL
                                                AND di.CallDisposition <> ''
                                                AND di.DateHour >= {cutoffdate!r} 
                                            GROUP BY di.CDI
                                        )
                                        SELECT 
                                            dexp.ExposureKey,
                                            MIN(attempt_dates_after_LTSA.DateOfFirstAttempt) AS DateOfFirstAttempt, 
                                            MAX(attempt_dates_after_LTSA.DateOfLastAttempt) AS DateOfLastAttempt,
                                            COALESCE(SUM(attempt_dates_after_LTSA.TotalCountOfAttempts), 0) AS TotalCountOfAttempts_after_LTSA 
                                        FROM dexposure dexp 
                                        LEFT JOIN ATTEMPT_DATES_AFTER_LTSA attempt_dates_after_LTSA 
                                                ON dexp.CDI = attempt_dates_after_LTSA.CDI
                                        GROUP BY dexp.ExposureKey                
                                    """)

attempt_dates_after_LTSA.createOrReplaceTempView("attempt_dates_after_LTSA")

In [0]:
attempt_dates_before_LTSA = spark.sql(f"""
                                        WITH ATTEMPT_DATES_BEFORE_LTSA AS (
                                            SELECT 
                                                intr.CDI,
                                                CASE WHEN intr.PortfolioName = 'Mercury' THEN MIN(intr.InteractionDate) ELSE NULL END AS DateOfFirstAttempt,
                                                CASE WHEN intr.PortfolioName = 'Mercury' THEN MAX(intr.InteractionDate) ELSE NULL END AS DateOfLastAttempt,
                                                CASE WHEN intr.PortfolioName = 'Mercury' THEN COUNT(intr.InteractionKey) ELSE 0 END AS TotalCountOfAttempts
                                            FROM `{env_prefix}-cdh`.primary.interaction intr 
                                            WHERE   intr.BusinessDate in ({LastWorkingDate!r},'2023-12-29')
                                                AND intr.InteractionDate < {cutoffdate!r}
                                                AND intr.SourceSystem = 'qc'
                                                AND intr.ContactType IN ('Right Person Contact', '3rd Person Contact', 'No Contact','RIGHT PARTY CONTACT')

                                            GROUP BY intr.CDI, intr.PortfolioName
                                            UNION 
                                            SELECT
                                                di.CDI, 
                                                CASE WHEN di.PortfolioName = 'Mercury' THEN MIN(CAST(di.DateHour AS DATE)) ELSE NULL END AS DateOfFirstAttempt,
                                                CASE WHEN di.PortfolioName = 'Mercury' THEN MAX(CAST(di.DateHour AS DATE)) ELSE NULL END AS DateOfLastAttempt,
                                                CASE WHEN di.PortfolioName = 'Mercury' THEN COUNT(di.DialerKey) ELSE 0 END AS TotalCountOfAttempts
                                            FROM `{env_prefix}-cdh`.primary.dialer di
                                            WHERE di.BusinessDate = {LastWorkingDate!r}
                                                AND di.CallDisposition IS NOT NULL
                                                AND di.CallDisposition <> ''
                                                AND di.DateHour < {cutoffdate!r}
                                            GROUP BY di.CDI, di.PortfolioName
                                        )
                                        SELECT 
                                            dexp.ExposureKey,
                                            CASE WHEN dexp.PortfolioName = 'Mercury' THEN MIN(attempt_dates_before_LTSA.DateOfFirstAttempt) ELSE NULL END AS DateOfFirstAttempt, 
                                            CASE WHEN dexp.PortfolioName = 'Mercury' THEN MAX(attempt_dates_before_LTSA.DateOfLastAttempt) ELSE NULL END AS DateOfLastAttempt,
                                            CASE WHEN dexp.PortfolioName = 'Mercury' THEN COALESCE(SUM(attempt_dates_before_LTSA.TotalCountOfAttempts), 0) ELSE 0 END AS TotalCountOfAttempts_before_LTSA 
                                        FROM dexposure dexp 
                                        LEFT JOIN ATTEMPT_DATES_BEFORE_LTSA attempt_dates_before_LTSA 
                                                ON dexp.CDI = attempt_dates_before_LTSA.CDI
                                        GROUP BY dexp.ExposureKey, dexp.PortfolioName
                                    """)

attempt_dates_before_LTSA.createOrReplaceTempView("attempt_dates_before_LTSA")

In [0]:
total_attempts = spark.sql(f"""
                                WITH TOTAL_ATTEMPTS AS (
                                    SELECT 
                                        intr.CDI,
                                        intr.InteractionDate AS AttemptDate,
                                        intr.InteractionKey AS AttemptKey
                                    FROM `{env_prefix}-cdh`.primary.interaction intr 
                                    WHERE   intr.BusinessDate in ({LastWorkingDate!r},'2023-12-29')
                                        AND intr.SourceSystem = 'qc'
                                        AND intr.ContactType IN ('Right Person Contact', '3rd Person Contact', 'No Contact','RIGHT PARTY CONTACT')
                                    UNION 
                                    SELECT
                                        di.CDI, 
                                        CAST(di.DateHour AS DATE) AS AttemptDate,
                                        di.DialerKey AS AttemptKey
                                    FROM `{env_prefix}-cdh`.primary.dialer di
                                    WHERE di.BusinessDate = {LastWorkingDate!r}
                                        AND di.CallDisposition IS NOT NULL
                                        AND di.CallDisposition <> ''
                                )
                                SELECT 
                                    dexp.ExposureKey,
                                    COUNT(DISTINCT total_attempts.AttemptDate) AS DistinctAttemptDays,
                                    COALESCE(COUNT(total_attempts.AttemptKey), 0) AS TotalCountOfAttempts
                                FROM dexposure dexp 
                                LEFT JOIN TOTAL_ATTEMPTS total_attempts 
                                        ON dexp.CDI = total_attempts.CDI
                                GROUP BY dexp.ExposureKey                
                                    """)

total_attempts.createOrReplaceTempView("total_attempts")

In [0]:
guarantor_attempts = spark.sql(f"""
                                    WITH GUARANTOR_ATTEMPTS AS (
                                        SELECT 
                                            intr.CDI,
                                            MIN(intr.InteractionDate) AS DateOfFirstGuarantorAttempt,
                                            MAX(intr.InteractionDate) AS DateOfLastGuarantorAttempt,
                                            COUNT(intr.InteractionKey) AS TotalCountOfGuarantorAttempts
                                        FROM `{env_prefix}-cdh`.primary.interaction intr 
                                        INNER JOIN `{env_prefix}-cdh`.primary.exposure exp
                                            ON intr.ExposureKey = exp.ExposureKey
                                                AND intr.CDI = exp.CDI
                                                AND exp.PartyRelationshipType != 'Primary Owner'
                                        WHERE   intr.BusinessDate in ({LastWorkingDate!r},'2023-12-29')
                                            AND intr.InteractionDate >= {cutoffdate!r}
                                            AND intr.SourceSystem = 'qc'
                                            AND intr.ContactType IN ('Right Person Contact', '3rd Person Contact', 'No Contact','RIGHT PARTY CONTACT')
                                        GROUP BY intr.CDI
                                        UNION 
                                        SELECT
                                            di.CDI, 
                                            MIN(CAST(di.DateHour AS DATE)) AS DateOfFirstGuarantorAttempt,
                                            MAX(CAST(di.DateHour AS DATE)) AS DateOfLastGuarantorAttempt,
                                            COUNT(di.DialerKey) AS TotalCountOfGuarantorAttempts
                                        FROM `{env_prefix}-cdh`.primary.dialer di
                                        INNER JOIN `{env_prefix}-cdh`.primary.exposure exp
                                            ON  di.CDI = exp.CDI
                                                AND exp.PartyRelationshipType != 'Primary Owner'
                                        WHERE di.BusinessDate = {LastWorkingDate!r}
                                                AND di.CallDisposition IS NOT NULL
                                                AND di.CallDisposition <> ''
                                                AND di.DateHour >= {cutoffdate!r}
                                        GROUP BY di.CDI
                                    )
                                    SELECT 
                                        dexp.ExposureKey,
                                        MIN(guarantor_attempts.DateOfFirstGuarantorAttempt) AS DateOfFirstGuarantorAttempt, 
                                        MAX(guarantor_attempts.DateOfLastGuarantorAttempt) AS DateOfLastGuarantorAttempt,
                                        COALESCE(SUM(guarantor_attempts.TotalCountOfGuarantorAttempts), 0) AS TotalCountOfGuarantorAttempts, 
                                        COUNT(DISTINCT guarantor_attempts.CDI) AS DistinctCountAttemptedGuarantors
                                    FROM dexposure dexp 
                                    INNER JOIN `{env_prefix}-cdh`.primary.exposure exp
                                            ON dexp.ExposureKey = exp.ExposureKey
                                                AND exp.PartyRelationshipType != 'Primary Owner'
                                    LEFT JOIN GUARANTOR_ATTEMPTS guarantor_attempts 
                                            ON exp.CDI = guarantor_attempts.CDI
                                    GROUP BY dexp.ExposureKey                
                                    """)

guarantor_attempts.createOrReplaceTempView("guarantor_attempts")

In [0]:
contact_dates = spark.sql(f"""
                            WITH CONTACT_DATES AS (
                                SELECT 
                                    intr.CDI,
                                    MIN(intr.InteractionDate) AS DateOfFirstContact,
                                    MAX(intr.InteractionDate) AS DateOfLastContact,
                                    COUNT(intr.InteractionKey) AS TotalCountOfContacts
                                FROM `{env_prefix}-cdh`.primary.interaction intr 
                                WHERE  intr.BusinessDate in ({LastWorkingDate!r},'2023-12-29')
                                    AND intr.InteractionDate >= {cutoffdate!r}
                                    AND intr.SourceSystem = 'qc'
                                    AND intr.ContactType IN ('Right Person Contact', '3rd Person Contact','RIGHT PARTY CONTACT')
                                GROUP BY intr.CDI
                                UNION 
                                SELECT
                                    di.CDI, 
                                    MIN(CAST(di.DateHour AS DATE)) AS DateOfFirstContact,
                                    MAX(CAST(di.DateHour AS DATE)) AS DateOfLastContact,
                                    COUNT(di.DialerKey) AS TotalCountOfContacts
                                FROM `{env_prefix}-cdh`.primary.dialer di
                                WHERE di.BusinessDate = {LastWorkingDate!r}
                                    AND di.DateHour >= {cutoffdate!r}
                                    AND di.CallDisposition IN ('ANSWERED', 'ANSWER')
                                GROUP BY di.CDI
                            )
                            SELECT 
                                dexp.ExposureKey,
                                MIN(contact_dates.DateOfFirstContact) AS DateOfFirstContact, 
                                MAX(contact_dates.DateOfLastContact) AS DateOfLastContact,
                                COALESCE(SUM(contact_dates.TotalCountOfContacts), 0) AS TotalCountOfContacts 
                            FROM dexposure dexp 
                            LEFT JOIN CONTACT_DATES contact_dates 
                                    ON dexp.CDI = contact_dates.CDI
                            GROUP BY dexp.ExposureKey                
                                    """)

contact_dates.createOrReplaceTempView("contact_dates")

### RPCs

In [0]:
rpc_all = spark.sql(f"""
                        SELECT 
                            intr.ExposureKey,
                            MIN(intr.InteractionDate) AS DateOfFirstRPC,
                            MAX(intr.InteractionDate) AS DateOfLastRPC,
                            COUNT(intr.InteractionKey) AS TotalCountOfRPCs
                        FROM `{env_prefix}-cdh`.primary.interaction intr 
                        WHERE   intr.BusinessDate = {LastWorkingDate!r}
                            --AND intr.InteractionDate >= '{cutoffdate!r}'
                            AND intr.SourceSystem = 'qc'
                            AND intr.ContactType in  ('Right Person Contact','RIGHT PARTY CONTACT')
                        GROUP BY intr.ExposureKey         
                            """)

rpc_all.createOrReplaceTempView("rpc_all")

In [0]:
rpc_before_ltsa = spark.sql(f"""
                                SELECT
                                    intr.ExposureKey, intr.PortfolioName,
                                    CASE WHEN intr.PortfolioName = 'Mercury' THEN MAX(intr.InteractionDate) ELSE NULL END AS LastDateOfRPCsBeforeLTSA,
                                    CASE WHEN intr.PortfolioName = 'Mercury' THEN COUNT(intr.InteractionKey) ELSE 0 END AS TotalCountOfRPCsBeforeLTSA
                                FROM `{env_prefix}-cdh`.primary.interaction intr
                                WHERE   intr.BusinessDate = {LastWorkingDate!r}
                                    AND intr.InteractionDate < {cutoffdate!r}
                                    AND intr.SourceSystem = 'qc'
                                    AND intr.ContactType in ('Right Person Contact','RIGHT PARTY CONTACT')
                                GROUP BY intr.ExposureKey, intr.PortfolioName
                            """)

rpc_before_ltsa.createOrReplaceTempView("rpc_before_ltsa")

In [0]:
rpc_last_status = spark.sql(f"""
                                SELECT DISTINCT
                                    intr.ExposureKey, 
                                    intr.AttemptOutcome
                                FROM `{env_prefix}-cdh`.primary.interaction intr 
                                INNER JOIN rpc_all
                                    ON rpc_all.ExposureKey = intr.ExposureKey
                                    AND rpc_all.DateOfLastRPC = intr.InteractionDate
                                WHERE   intr.BusinessDate = {LastWorkingDate!r}
                                    AND intr.SourceSystem = 'qc'
                                    AND intr.ContactType in ('Right Person Contact','RIGHT PARTY CONTACT')   
                                    and intr.PortfolioName = {portfolioName!r}

                            """)

rpc_last_status.createOrReplaceTempView("rpc_last_status")

In [0]:
guarantor_RPC = spark.sql(f"""
                            SELECT 
                                intr.ExposureKey,
                                MIN(intr.InteractionDate) AS DateOfFirstGuarantorRPC,
                                MAX(intr.InteractionDate) AS DateOfLastGuarantorRPC,
                                COUNT(intr.InteractionKey) AS TotalCountOfGuarantorRPC, 
                                COUNT(DISTINCT intr.InteractionDate) AS TotalDistinctDatesOfGuarantorRPC
                            FROM `{env_prefix}-cdh`.primary.interaction intr 
                            INNER JOIN `{env_prefix}-cdh`.primary.exposure exp
                                ON intr.ExposureKey = exp.ExposureKey
                                    AND intr.CDI = exp.CDI
                                    AND exp.PartyRelationshipType != 'Primary Owner'
                            WHERE   intr.BusinessDate = {LastWorkingDate!r}
                                AND intr.SourceSystem = 'qc'
                                AND intr.ContactType in  ('Right Person Contact','RIGHT PARTY CONTACT')
                            GROUP BY intr.ExposureKey               
                            """)

guarantor_RPC.createOrReplaceTempView("guarantor_RPC")

In [0]:
count_RPC_per_status = spark.sql(f"""
                                    SELECT 
                                        intr.ExposureKey,
                                        COUNT(CASE WHEN AttemptOutcome = 'positive' THEN 1 END) AS PositiveRPC,
                                        COUNT(CASE WHEN AttemptOutcome = 'negative' THEN 1 END) AS NegativeRPC,
                                        COUNT(CASE WHEN AttemptOutcome NOT IN ('positive', 'negative') THEN 1 END) AS OtherRPC
                                    FROM `{env_prefix}-cdh`.primary.interaction intr 
                                    WHERE   intr.BusinessDate = {LastWorkingDate!r}
                                        AND intr.SourceSystem = 'qc'
                                        AND intr.ContactType in  ('Right Person Contact','RIGHT PARTY CONTACT')
                                    GROUP BY intr.ExposureKey               
                            """)

count_RPC_per_status.createOrReplaceTempView("count_RPC_per_status")

In [0]:
last_inbound_date = spark.sql(f"""
                                SELECT 
                                    intr.ExposureKey,
                                    MAX(intr.InteractionDate) AS LastInboundDate
                                FROM `{env_prefix}-cdh`.primary.interaction intr 
                                WHERE   intr.BusinessDate = {LastWorkingDate!r}
                                    AND intr.SourceSystem = 'qc'
                                    AND intr.InteractionTypeCode IN ('1020','6028','3070','3086','3094')
                                GROUP BY intr.ExposureKey               
                            """)

last_inbound_date.createOrReplaceTempView("last_inbound_date")

In [0]:
action_before_ltsa = spark.sql(f"""
                                WITH ACTION_BEFORE_LTSA AS (
                                    SELECT 
                                        intr.ExposureKey,
                                        intr.InteractionKey, 
                                        intr.ContactType
                                    FROM `{env_prefix}-cdh`.primary.interaction intr 
                                    WHERE   intr.BusinessDate = {LastWorkingDate!r}
                                        AND intr.InteractionDate < {cutoffdate!r}
                                        AND intr.SourceSystem = 'qc'
                                        AND intr.PortfolioName = 'Mercury'
                                        AND intr.ContactType IN ('Right Person Contact', '3rd Person Contact', 'No Contact','RIGHT PARTY CONTACT')
                                )
                                SELECT 
                                    dexp.ExposureKey,
                                    CASE 
                                        WHEN (SELECT COUNT(ACTION_BEFORE_LTSA.InteractionKey) FROM ACTION_BEFORE_LTSA WHERE ACTION_BEFORE_LTSA.ContactType in  ('Right Person Contact','RIGHT PARTY CONTACT')) > 1 THEN 1 
                                        ELSE 0 
                                    END AS RPC_before_LTSA,
                                    CASE 
                                        WHEN (SELECT COUNT(ACTION_BEFORE_LTSA.InteractionKey) FROM ACTION_BEFORE_LTSA ) > 1 THEN 1 
                                        ELSE 0 
                                    END AS Attempt_before_LTSA
                                FROM dexposure dexp 
                                LEFT JOIN ACTION_BEFORE_LTSA  
                                        ON dexp.ExposureKey = ACTION_BEFORE_LTSA.ExposureKey
                                    """)

action_before_ltsa.createOrReplaceTempView("action_before_ltsa")

In [0]:
last_RPC_status_before_ltsa = spark.sql(f"""
                                        WITH LAST_RPC_STATUS_BEFORE_LTSA AS (
                                            SELECT 
                                                intr.ExposureKey,
                                                MAX(intr.InteractionDate) AS LastRPCDate
                                            FROM `{env_prefix}-cdh`.primary.interaction intr 
                                            WHERE   intr.BusinessDate = {LastWorkingDate!r}
                                                AND intr.InteractionDate < {cutoffdate!r}
                                                AND intr.SourceSystem = 'qc'
                                                AND intr.PortfolioName = 'Mercury'
                                                AND intr.ContactType in  ('Right Person Contact','RIGHT PARTY CONTACT')
                                                AND intr.interactionTypeCode in 
                                                ( '2001','2002','2003','2004','2005','2006','2007','2008','2009','2010','2011','2013','2014','2015','3007','2016'
,'2017','2018','2019','2020','2021','2023','2025','2026','2028','2029','3008','3009','3046','3048','5009','2012','2030','3021','3022','3023'
,'3024','3030','3039','3058','3059','3060','3061','3072','3082','3083','3084','3085','3087','3088','3090','3095','3123','3124','3125','3126'
,'3127','3130','3161','3162','3163','3164','3165','3192','3195','3196','3197','3198','3200','3201','3202','3203','3204','3210','3211','3212'
,'3213','3214','3215','3216','3218','5481','5491','5494','6102','6103','7019','7020','7021','7022','7023','7024'
,'7025','7026','7027','7028','7029','7035','8003','8004','8008','3001','3005','3013','3015','3016','3028','3031','3032','3036','6028')
                                            GROUP BY intr.ExposureKey
                                        )
                                        SELECT 
                                            dexp.ExposureKey,
                                            sub.AttemptOutcome
                                        FROM dexposure dexp 
                                        LEFT JOIN 
                                            (SELECT intr.ExposureKey, 
                                                    intr.AttemptOutcome
                                            FROM LAST_RPC_STATUS_BEFORE_LTSA
                                            INNER JOIN `{env_prefix}-cdh`.primary.interaction intr
                                                ON LAST_RPC_STATUS_BEFORE_LTSA.ExposureKey = intr.ExposureKey
                                            WHERE intr.InteractionDate = LAST_RPC_STATUS_BEFORE_LTSA.LastRPCDate
                                                AND intr.SourceSystem = 'qc'
                                                AND intr.PortfolioName = 'Mercury'
                                                AND intr.ContactType in  ('Right Person Contact','RIGHT PARTY CONTACT')
                                                ) sub  

                                                ON dexp.ExposureKey = sub.ExposureKey
                                    """)

last_RPC_status_before_ltsa.createOrReplaceTempView("last_RPC_status_before_ltsa")

In [0]:
guarantor_RPC_before_LTSA = spark.sql(f"""
                                        WITH GUARANTOR_RPC_BEFORE_LTSA AS (
                                            SELECT 
                                                intr.ExposureKey,
                                                MAX(intr.InteractionDate) AS LastDateGuarantorRPCBeforeLTSA,
                                                COUNT(intr.InteractionKey) AS TotalGuarantorRPCBeforeLTSA
                                            FROM `{env_prefix}-cdh`.primary.interaction intr 
                                            INNER JOIN `{env_prefix}-cdh`.primary.exposure exp
                                                ON intr.ExposureKey = exp.ExposureKey
                                                    AND intr.CDI = exp.CDI
                                                    AND exp.PartyRelationshipType != 'Primary Owner'
                                            WHERE   intr.BusinessDate = {LastWorkingDate!r}
                                                AND intr.SourceSystem = 'qc'
                                                AND intr.ContactType in  ('Right Person Contact','RIGHT PARTY CONTACT')
                                                AND intr.InteractionDate < {cutoffdate!r}
                                                AND intr.PortfolioName = 'Mercury'
                                            GROUP BY intr.ExposureKey
                                        )
                                        SELECT 
                                            dexp.ExposureKey, 
                                            CASE 
                                                WHEN GUARANTOR_RPC_BEFORE_LTSA.TotalGuarantorRPCBeforeLTSA > 1 THEN 1 
                                                ELSE 0 
                                            END AS Guarantor_RPC_before_LTSA
                                        FROM dexposure dexp 
                                        LEFT JOIN GUARANTOR_RPC_BEFORE_LTSA  
                                            ON dexp.ExposureKey = GUARANTOR_RPC_BEFORE_LTSA.ExposureKey
                            """)

guarantor_RPC_before_LTSA.createOrReplaceTempView("guarantor_RPC_before_LTSA")

In [0]:
contacts_final_tbl = spark.sql(f"""
                            SELECT DISTINCT dexposure.ExposureKey,
                                    eap.EAP_Status,
                                    eap.Last_EAP_Date,
                                    re.Positive_RE_Exists,
                                    neg_attempts.TotalCountOfNegativeAttempts AS TotalCountOfOffersRejected,
                                    neg_attempts.DateOfLastNegativeAttempt AS DateofLastRejectedOffer,
                                    neg_reason.InteractionComments AS RejectionReason,
                                    ATTEMPT_DATES_AFTER_LTSA.DateOfFirstAttempt,
                                    ATTEMPT_DATES_AFTER_LTSA.DateOfLastAttempt AS DateOfLastAttemptAfterLTSA,
                                    ATTEMPT_DATES_AFTER_LTSA.TotalCountOfAttempts_after_LTSA,
                                    TOTAL_ATTEMPTS.TotalCountOfAttempts,
                                    TOTAL_ATTEMPTS.DistinctAttemptDays,
                                    ATTEMPT_DATES_BEFORE_LTSA.DateOfLastAttempt AS DateOfLastAttemptBeforeLTSA,
                                    GUARANTOR_ATTEMPTS.TotalCountOfGuarantorAttempts,
                                    GUARANTOR_ATTEMPTS.DateOfLastGuarantorAttempt,
                                    GUARANTOR_ATTEMPTS.DistinctCountAttemptedGuarantors,
                                    CONTACT_DATES.DateOfFirstContact,
                                    CONTACT_DATES.DateOfLastContact,
                                    CONTACT_DATES.TotalCountOfContacts,
                                    RPC_ALL.TotalCountOfRPCs,
                                    RPC_LAST_STATUS.AttemptOutcome AS Last_RPC_Status,
                                    RPC_ALL.DateOfLastRPC,
                                    GUARANTOR_RPC.TotalCountOfGuarantorRPC,
                                    GUARANTOR_RPC.DateOfLastGuarantorRPC,
                                    COUNT_RPC_PER_STATUS.PositiveRPC,
                                    COUNT_RPC_PER_STATUS.NegativeRPC,
                                    COUNT_RPC_PER_STATUS.OtherRPC,
                                    RPC_BEFORE_LTSA.LastDateOfRPCsBeforeLTSA,
                                    LAST_INBOUND_DATE.LastInboundDate,
                                    ACTION_BEFORE_LTSA.Attempt_before_LTSA,
                                    ACTION_BEFORE_LTSA.RPC_before_LTSA,
                                    GUARANTOR_RPC_BEFORE_LTSA.Guarantor_RPC_before_LTSA,
                                    LAST_RPC_STATUS_BEFORE_LTSA.AttemptOutcome AS LastRPCStatusBeforeLTSA
                                    FROM dexposure
                                    LEFT JOIN eap
                                    ON dexposure.ExposureKey = eap.ExposureKey
                                    LEFT JOIN re
                                    ON dexposure.ExposureKey = re.ExposureKey
                                    LEFT JOIN neg_attempts
                                    ON dexposure.ExposureKey = neg_attempts.ExposureKey
                                    LEFT JOIN neg_reason
                                    ON dexposure.ExposureKey = neg_reason.ExposureKey
                                    LEFT JOIN ATTEMPT_DATES_AFTER_LTSA
                                    ON dexposure.ExposureKey = ATTEMPT_DATES_AFTER_LTSA.ExposureKey
                                    LEFT JOIN TOTAL_ATTEMPTS
                                    ON dexposure.ExposureKey = TOTAL_ATTEMPTS.ExposureKey
                                    LEFT JOIN ATTEMPT_DATES_BEFORE_LTSA
                                    ON dexposure.ExposureKey = ATTEMPT_DATES_BEFORE_LTSA.ExposureKey
                                    LEFT JOIN GUARANTOR_ATTEMPTS
                                    ON dexposure.ExposureKey = GUARANTOR_ATTEMPTS.ExposureKey
                                    LEFT JOIN CONTACT_DATES
                                    ON dexposure.ExposureKey = CONTACT_DATES.ExposureKey
                                    LEFT JOIN RPC_ALL
                                    ON dexposure.ExposureKey = RPC_ALL.ExposureKey
                                    LEFT JOIN RPC_LAST_STATUS
                                    ON dexposure.ExposureKey = RPC_LAST_STATUS.ExposureKey
                                    LEFT JOIN GUARANTOR_RPC
                                    ON dexposure.ExposureKey = GUARANTOR_RPC.ExposureKey
                                    LEFT JOIN COUNT_RPC_PER_STATUS
                                    ON dexposure.ExposureKey = COUNT_RPC_PER_STATUS.ExposureKey
                                    LEFT JOIN RPC_BEFORE_LTSA
                                    ON dexposure.ExposureKey = RPC_BEFORE_LTSA.ExposureKey
                                    LEFT JOIN LAST_INBOUND_DATE
                                    ON dexposure.ExposureKey = LAST_INBOUND_DATE.ExposureKey
                                    LEFT JOIN ACTION_BEFORE_LTSA
                                    ON dexposure.ExposureKey = ACTION_BEFORE_LTSA.ExposureKey
                                    LEFT JOIN GUARANTOR_RPC_BEFORE_LTSA
                                    ON dexposure.ExposureKey = GUARANTOR_RPC_BEFORE_LTSA.ExposureKey
                                    LEFT JOIN LAST_RPC_STATUS_BEFORE_LTSA
                                    ON dexposure.ExposureKey = LAST_RPC_STATUS_BEFORE_LTSA.ExposureKey
                        """)

contacts_final_tbl.createOrReplaceTempView("contacts_final_tbl")

## Final Table

In [0]:
finaltbl1 = spark.sql(f"""
                    SELECT DISTINCT de.PortfolioName AS Portfolio,
                                de.CDI AS BorrowerID, de.ExposureKey AS LoanID,
                                de.AssetSegment AS AssetClass,
                                tbr.BalanceRange, de.InitialAssignmentAmount AS TLCAtMigrationDate,
                                de.OutstandingTotalBalance AS TotalLegalClaim, de.TotalPrincipal AS PrincipalBalance, de.TotalInterest AS Interest, de.TotalExpense AS Expense, de.TotalAccruals AS Accrued,
                                col.MonthlyPayments, col.TotalPaymentsAfterLTSA, col.LastPaymentDate,
                                de.ExposureStatus AS LoanStatus,
                                CASE WHEN de.HasActiveSettlementFlag IS TRUE THEN de.SettlementBucket ELSE de.Bucket END AS Current_Month_Bucket,
                                CASE WHEN de.IsActiveFlag IS FALSE THEN 'Y' ELSE 'N' END AS ClosedAccounts,
                                CAST(de.ClosedDate AS DATE) AS ClosedAccountsDate,
                                assgn.AssignedChannel,
                                dp.Segment,
                                dpt.WorkableFlag AS WorkableFlag,
                                dpt.NonWorkableReason AS NonWorkableReason,
                                CASE WHEN de.PortfolioName = 'Mercury' THEN cn.campaignname ELSE NULL END AS CampaignName,
                                prt.subsegment AS Leverage,
                                de.CoCStatus,
                                CASE WHEN de.DenouncementIndicator IS TRUE THEN 'Yes' ELSE 'No' END AS Denounced,
                                CASE WHEN de.LPSEng IS NOT NULL THEN 'Yes' ELSE 'No' END AS CaseUnderLegal,
                                de.LPSEng AS AccountLPS, CAST(de.LPSDate AS DATE) AS AccountLPSDate,
                                CASE WHEN SUBSTRING(de.ProductKey, CHARINDEX(';', de.ProductKey) + 1, LEN(de.ProductKey)) IN ('6016', '6031', '6900', '6902', '6903', '6904', '7424') THEN 'Yes' ELSE 'No' END AS AutoLoanFlag,
                                COALESCE(
                                        CASE 
                                            WHEN collm.CollateralType = 'RE' 
                                                THEN collm.RealEstateCollateralStatus 
                                            ELSE collm.CollateralStatus 
                                        END,
                                        'Inactive'
                                    ) AS CollateralExist,
                                coll.MinCollateralLien,
                                coll.CollateralTotalAmount,
                                coll.MinCollateralLienTotalAmount,
                                sttl.DueAmount,
                                sttl.KeptRate,
                                sttl.ImplementedDate,
                                sttl.CountofImplementationsPerAccount,
                                sttl.AppliedDate,
                                sttl.SettlementAmount,
                                sttl.HaircutAmount, sttl.HaircutPercentage, sttl.HaircutPrincipalPercentage, sttl.TotalAmountDueAfterHaircut,
                                sttl.SettlementType, sttl.ApplicationType,
                                sttl.Status, sttl.SettlementFromMigration,
                                sttl.NumberOfInstallments, sttl.NumberOfPastInstallments, sttl.NumberOfMissedInstallments, sttl.ShortTerm_LongTermFlag, sttl.FirstInstallmentDate, sttl.MonthlyInstallmentAmount,
                                dp.L3869Status AS L3869StatusHistory,
                                law.L3869Status1 AS L3869Status,
                                law.L3869MonthExit,
                                law.L3869CourtDecision,
                                law.L3869FinalCourtDate,
                                law.PreviousSegmentation,
                                law.L3869SubmissionDate,
                                law.L3869ResignationDate,
                                law.EkptosiFlag,
                                law.EkptosiStatus,
                                law.`L3869MLΑmount`,
                                law.MLinABExist,
                                totaldebt.L3869TotalDebt,
                                de.DaysPastDueWithMateriality AS DaysPastDue, de.OffBalance,
                                CASE WHEN lf.LetterFlagCDI IS NOT NULL THEN 'Yes' ELSE 'No' END AS LetterFlag
                    FROM dexposure de
                    LEFT JOIN TaxBalanceRange tbr
                    ON de.CDI = tbr.CDI
                    LEFT JOIN Collections col
                    ON regexp_replace(de.ExposureKey, '[^0-9]', '') = regexp_replace(col.ExposureKey, '[^0-9]', '')
                    AND de.PortfolioName = col.PortfolioName
                    LEFT JOIN assignment assgn
                    ON regexp_replace(de.ExposureKey, '[^0-9]', '') = regexp_replace(assgn.ExposureKey, '[^0-9]', '')
                    AND de.PortfolioName = assgn.PortfolioName
                    LEFT JOIN dparty dp
                    ON de.CDI = dp.CDI
                    LEFT JOIN party prt
                    ON de.PartyTaxRegistrationNumber = prt.TaxRegistrationNumber
                    AND de.PortfolioName = prt.PortfolioName
                    LEFT JOIN CollateralMappingTempLien coll
                    ON de.ExposureKey = coll.ExposureKey
                    LEFT JOIN CollateralMapping collm
                    on collm.ExposureKey = de.ExposureKey
                    LEFT JOIN settlement_fields sttl
                    ON de.ExposureKey = sttl.ExposureKey
                    LEFT JOIN dunderlaw law
                    ON de.CDI = law.CDI
                    LEFT JOIN l3869totaldebt totaldebt
                    ON de.ExposureKey = totaldebt.ExposureKey
                    AND de.CDI = totaldebt.CDI
                    LEFT JOIN letterflagexist lf
                    ON de.CDI = lf.LetterFlagCDI
                    LEFT JOIN DPortfolioToolThirdParties dpt
                    ON dpt.exposurekey=de.exposurekey
                    LEFT JOIN campaignname cn
                    ON de.PortfolioName = 'Mercury'
                    AND de.ExposureKey = cn.AccountNumber
""")
 
finaltbl1.createOrReplaceTempView("finaltbl1")

In [0]:
finaltbl = spark.sql(f"""
                    SELECT DISTINCT A.*,
                            B.EAP_Status,
                            B.Last_EAP_Date,
                            B.Positive_RE_Exists,
                            B.TotalCountOfOffersRejected,
                            B.DateofLastRejectedOffer,
                            B.RejectionReason,
                            B.DateOfFirstAttempt,
                            B.DateOfLastAttemptAfterLTSA,
                            B.TotalCountOfAttempts_after_LTSA,
                            B.TotalCountOfAttempts,
                            B.DistinctAttemptDays,
                            B.DateOfLastAttemptBeforeLTSA,
                            B.TotalCountOfGuarantorAttempts,
                            B.DateOfLastGuarantorAttempt,
                            B.DistinctCountAttemptedGuarantors,
                            B.DateOfFirstContact,
                            B.DateOfLastContact,
                            B.TotalCountOfContacts,
                            B.TotalCountOfRPCs,
                            B.Last_RPC_Status,
                            B.DateOfLastRPC,
                            B.TotalCountOfGuarantorRPC,
                            B.DateOfLastGuarantorRPC,
                            B.PositiveRPC,
                            B.NegativeRPC,
                            B.OtherRPC,
                            B.LastDateOfRPCsBeforeLTSA,
                            B.LastInboundDate,
                            B.Attempt_before_LTSA,
                            B.RPC_before_LTSA,
                            B.Guarantor_RPC_before_LTSA,
                            B.LastRPCStatusBeforeLTSA
                    FROM finaltbl1 A
                    LEFT JOIN contacts_final_tbl B
                    ON A.LoanID = B.ExposureKey
""")

finaltbl.createOrReplaceTempView("finaltbl")

## Report Export

In [0]:
final_report_df = finaltbl.toPandas()

# Define the temporary file and output path
temp_file = f"/tmp/{output_file_prefix}{now_datetime_str}.xlsx"

output_buffer = BytesIO()
with pd.ExcelWriter(output_buffer) as writer:
    final_report_df.to_excel(writer, sheet_name=f"{RefMonth.month}.{RefMonth.year}", index=False)

with open(temp_file, "wb") as f:
    f.write(output_buffer.getvalue())

output_path = f"{report_output}{run_date_str}/{output_file_prefix}{LastWorkingDate}.xlsx"

# Ensure the directory exists
dbutils.fs.mkdirs(f"{report_output}{run_date_str}")

# Delete the output path if it exists
if os.path.exists(output_path):
    os.remove(output_path)

# Copy the temporary file to the final output path
shutil.copy(temp_file, output_path)

print(f"Output file is written successfully: {output_path}")